### Собери современный гибридный трансформер: Qwen3.5-0.8B

Реализуй в PyTorch **текстовую основу Qwen3.5-0.8B**. Это гибридная архитектура: большинство слоёв используют **Gated DeltaNet** (рекуррентный delta-rule без квадратичной матрицы внимания), а каждый четвёртый слой использует **полное внимание** с частичным RoPE и выходным затвором.

В обязательной части реализуй обычный multi-head attention с одинаковым числом голов query, key и value. Настоящий чекпоинт использует grouped-query attention (**GQA**) — это дополнительное задание.

Заполни методы `TODO` в **`qwen.py`** и сравни каждый блок с аналогичным по форме модулем `transformers`, скопировав веса.

Обязательная часть: **разделы 2–9 и конвертер конфигурации Hugging Face**. Три независимых необязательных задания: GQA, блочная рекуррентность и Multi-Token Prediction (MTP). GQA и другие способы ускорения объясняются в лекции **week07_efficiency**.

Тесты маленькой модели работают на CPU без скачивания весов. Интеграционные проверки на настоящем чекпоинте включаются отдельно.

Область работы: **только текст**, без KV cache и padding-масок. Forward-проходы и предоставленная GDN-рекуррентность должны поддерживать autograd. `torch.no_grad()` используй только в местах инференса.

> Неофициальный русский перевод [исходного задания](./homework.ipynb). Код и тесты сохранены.

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import transformers
import qwen

#### 1. Конфигурация

Класс \`Qwen35Config\` уже предоставлен. Тип каждого слоя вычисляется по формуле:

$$
\text{layer\_types}[i] =
\begin{cases}
\text{"full\_attention"} & \text{if } (i+1) \bmod \text{interval} = 0 \\
\text{"linear\_attention"} & \text{otherwise}
\end{cases}
$$

Так же вычисляет типы слоёв настоящая \`Qwen3_5TextConfig\`. Поэтому при совпадении \`num_hidden_layers\` и \`full_attention_interval\` набор слоёв будет идентичен, что позволяет загружать веса из реальной модели той же размерности.

Маленькая конфигурация использует уменьшенные размерности для CPU и задаёт \`num_attention_heads == num_key_value_heads\` для обычного multi-head attention. В обязательных заданиях число голов меняется, но количество query- и KV-голов совпадает; другие соотношения относятся к бонусу GQA.

При этом отношение голов key/value внутри Gated DeltaNet остаётся частью обязательной работы.

In [ ]:
small_config = qwen.make_small_config()
print(small_config)
print("\nlayer_types:", small_config.layer_types)


#### 2. RMSNorm (вариант Qwen с нулевой инициализацией)

Вес инициализируется **нулём**. Выход равен \`normalized * (1 + weight)\`, а не \`weight * normalized\`:

$$
\hat{x} = \frac{x}{\sqrt{\operatorname{mean}(x^2,\ \text{dim}=-1) + \epsilon}}
$$

$$
\text{RMSNorm}(x) = \hat{x} \odot (1 + w)
$$

Реализуй \`RMSNorm.forward\` в \`qwen.py\`. Вычисляй нормировку и умножение на вес в float32, а результат преобразуй обратно к dtype входа. Это важно для float16 и bfloat16.

In [ ]:
norm = qwen.RMSNorm(small_config.hidden_size, eps=small_config.rms_norm_eps)
x = torch.randn(2, 5, small_config.hidden_size)
out = norm(x)
print("output shape:", out.shape, " (weight=0 at init, so this should just be normalized x)")
print("matches plain normalization:", torch.allclose(
    out, x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + norm.eps)
))

#### 3. Gated RMSNorm

Нормировка как у RMSNorm, но веса инициализируются **единицами**, а выход умножается на \`silu(gate)\`:

$$
\hat{x} = \frac{x}{\sqrt{\operatorname{mean}(x^2,\ \text{dim}=-1) + \epsilon}}
$$

$$
\text{RMSNormGated}(x, g) = w \odot \hat{x} \odot \operatorname{silu}(g)
$$

Реализуй \`RMSNormGated.forward\` в \`qwen.py\`.

Воспроизведи порядок округлений из зафиксированной эталонной реализации:
1. Нормализуй \`x.float()\` в float32.
2. Приведи нормализованный тензор обратно к \`x.dtype\`.
3. Умножь на \`self.weight\` в dtype параметра.
4. Умножь результат на \`silu(gate.float())\`, что переведёт промежуточный результат в float32.
5. Приведи выход к \`x.dtype\`.

Промежуточные преобразования важны для float16/bfloat16. В отличие от \`RMSNorm\` выше, этот вариант преобразует dtype **до** применения обучаемого масштаба.

In [ ]:
gated_norm = qwen.RMSNormGated(small_config.linear_value_head_dim, eps=small_config.rms_norm_eps)
x = torch.randn(2, 5, small_config.linear_value_head_dim)
gate = torch.randn(2, 5, small_config.linear_value_head_dim)
print("output shape:", gated_norm(x, gate).shape)

#### 4. Частичное вращательное позиционное кодирование (partial RoPE)

Вращаются только первые \`partial_rotary_factor * head_dim\` каналов q/k; остальные проходят без изменений.

$$
\text{inv\_freq}_i = \theta^{-2i / d_{\text{rot}}},
\qquad i = 0, \dots, d_{\text{rot}}/2 - 1
$$

$$
\text{freqs} = \text{position\_ids} \otimes \text{inv\_freq},
\qquad \text{emb} = [\text{freqs},\ \text{freqs}]
$$

$$
\cos,\ \sin \;=\; \cos(\text{emb}),\ \sin(\text{emb})
$$

$$
\text{rotate\_half}([x_1, x_2]) = [-x_2,\ x_1]
$$

$$
x_{\text{rot}}' = x_{\text{rot}} \odot \cos + \text{rotate\_half}(x_{\text{rot}}) \odot \sin
$$

Конструктор уже предоставлен. Реализуй \`Qwen35RotaryEmbedding.forward\`, \`rotate_half\` и \`apply_rotary_pos_emb\` в \`qwen.py\`. Точные формы тензоров смотри в \`tests/test_qwen_public.py::TestRotaryEmbedding\`.

In [ ]:
rope = qwen.Qwen35RotaryEmbedding(small_config)
seq_len = 10
position_ids = torch.arange(seq_len).unsqueeze(0)
dummy_hidden = torch.randn(1, seq_len, small_config.hidden_size)
cos, sin = rope(dummy_hidden, position_ids)
print("cos/sin shape:", cos.shape, "(batch, seq_len, rotary_dim) -- rotary_dim =", cos.shape[-1],
      "= head_dim * partial_rotary_factor =", small_config.head_dim, "*", small_config.partial_rotary_factor)

q = torch.randn(1, 4, seq_len, small_config.head_dim)
k = torch.randn(1, 4, seq_len, small_config.head_dim)
q_rot, k_rot = qwen.apply_rotary_pos_emb(q, k, cos, sin)
rotary_dim = cos.shape[-1]
print("pass-through channels untouched:", torch.equal(q_rot[..., rotary_dim:], q[..., rotary_dim:]))

#### 5. Полное внимание: обычное MHA + QK-нормировка по головам + частичное RoPE + выходной затвор

Реализуй обычный multi-head attention: на каждую голову внимания приходится по одной голове query, key и value (\`num_attention_heads == num_key_value_heads\`).

Добавь RMSNorm для q/k **по каждой голове**, частичное RoPE и sigmoid-затвор на выходе. Всё должно быть **векторизовано без циклов Python по головам**.

$$
Q = \text{q\_norm}(\text{q\_proj}(x)), \qquad
K = \text{k\_norm}(\text{k\_proj}(x)), \qquad
V = \text{v\_proj}(x)
$$

$$
\text{scores} = \frac{Q K^\top}{\sqrt{d_{\text{head}}}} + \text{mask}
$$

$$
\text{attn} = \operatorname{softmax}(\text{scores})\, V
$$

$$
\text{out} = \text{o\_proj}\big(\text{attn} \odot \sigma(\text{gate})\big)
$$

У $Q$, $K$, $V$ одинаковое число голов: каждой голове query непосредственно соответствует её key/value-голова. Затвор упакован в одну проекцию вместе с query.

Конструктор и вспомогательный код проекции QKV/gate предоставлены.

Реализуй \`GatedAttention.forward\` в \`qwen.py\` через батчевое умножение матриц и softmax:

- **Запрещены** циклы Python по головам и \`torch.nn.functional.scaled_dot_product_attention\` (SDPA).
- Учитывай аддитивную attention-маску.
- Если \`attention_mask is None\`, создай причинную маску; иначе используй переданную.
- Точные проекции и формы — \`tests/test_qwen_public.py::TestGatedAttention\`.

Встроенная диспетчеризация передаёт конфигурации с меньшим числом key/value-голов в необязательный \`GatedAttention.forward_gqa\`. Этот \`TODO\` **не требуется** в обязательной части. GQA объясняется на неделе 7; ниже есть бонус для желающих.

Предоставленный \`naive_looped_gated_attention\` делает такие же matmul/softmax, но отдельно для каждой головы. Используй его для проверки корректности и измерения скорости. Фиксированное ускорение по wall-clock не требуется: накладные расходы и CPU/GPU-зависимость различаются.

In [ ]:
layer_idx = small_config.layer_types.index("full_attention")
attn = qwen.GatedAttention(small_config, layer_idx)
attn.eval()

batch, seq_len = 2, 12
hidden_states = torch.randn(batch, seq_len, small_config.hidden_size)
mask = qwen.build_causal_mask(seq_len, hidden_states.dtype, hidden_states.device)
position_embeddings = qwen.Qwen35RotaryEmbedding(small_config)(
    hidden_states, torch.arange(seq_len).unsqueeze(0).expand(batch, -1)
)

with torch.no_grad():
    out_fast = attn(hidden_states, position_embeddings, mask)
    out_naive = qwen.naive_looped_gated_attention(attn, hidden_states, position_embeddings, mask)

print("output shape:", out_fast.shape)
print("matches naive per-head loop:", torch.allclose(out_fast, out_naive, atol=1e-4))

#### 6. SwiGLU MLP

Стандартная формула SwiGLU:

$$
\text{MLP}(x) = \text{down\_proj}\Big(
\operatorname{silu}\big(\text{gate\_proj}(x)\big) \odot \text{up\_proj}(x)
\Big)
$$

Здесь всё просто: реализуй \`MLP.forward\` в \`qwen.py\`.

In [ ]:
mlp = qwen.MLP(small_config)
x = torch.randn(2, 5, small_config.hidden_size)
print("output shape:", mlp(x).shape)

#### 7. Gated DeltaNet (линейное внимание)

Для каждой головы поддерживается рекуррентное состояние постоянного размера $S \in \mathbb{R}^{d_k \times d_v}$. Оно обновляется по **delta rule**: сравнить сохранённое значение с истинным, записать разницу с коэффициентом $\beta$ и применить затухание $g$:

$$
S_t \leftarrow S_{t-1} \cdot \exp(g_t)
$$

$$
\text{kv\_mem}_t = S_t^\top k_t
$$

$$
\delta_t = \big(v_t - \text{kv\_mem}_t\big) \cdot \beta_t
$$

$$
S_t \leftarrow S_t + k_t \otimes \delta_t
$$

$$
\text{out}_t = S_t^\top q_t
$$

$$
\beta_t = \sigma\big(\text{in\_proj\_b}(x_t)\big), \qquad
g_t = -\exp(A_{\log}) \cdot \operatorname{softplus}\big(\text{in\_proj\_a}(x_t) + \text{dt\_bias}\big)
$$

Конструктор и рекуррентная функция предоставлены. **Реализуй \`GatedDeltaNet.forward\`.**

Для входа формы \`(batch, seq_len, hidden_size)\` обозначим \`K = num_k_heads * head_k_dim\`, \`V = num_v_heads * head_v_dim\`.

1. Примени \`in_proj_qkv\`: результат \`(batch, seq_len, 2*K + V)\`. Транспонируй в \`(batch, 2*K + V, seq_len)\` для заданной depthwise \`conv1d\`. Симметричный padding даёт лишние временные шаги: оставь только **первые \`seq_len\` выходов**, чтобы результат зависел лишь от текущего и прошлого. Примени SiLU и транспонируй обратно.
2. Разбей последнюю размерность на **\`K\`, \`K\`, \`V\`** в порядке query, key, value. Измени форму query/key на \`(batch, seq_len, num_k_heads, head_k_dim)\`, а value — на \`(batch, seq_len, num_v_heads, head_v_dim)\`.
3. При \`num_v_heads > num_k_heads\` повтори каждую голову query/key **подряд** \`num_v_heads // num_k_heads\` раз. Все три тензора, переданные рекуррентной функции, должны иметь \`num_v_heads\` голов. Размерности key и value могут отличаться.
4. Из **исходных hidden states** вычисли \`beta = sigmoid(in_proj_b(x))\` и \`g\` по формуле затухания выше. Оба тензора имеют форму \`(batch, seq_len, num_v_heads)\`. Вычисляй затухание в float32, включая \`exp(A_log.float())\` и \`softplus(in_proj_a(x).float() + dt_bias.float())\`. И \`A_log\`, и \`dt_bias\` существенны: нельзя заменять их константами.
5. Вызови предоставленную \`gated_delta_rule_recurrent(query, key, value, g, beta)\`. Она уже L2-нормализует query/key, масштабирует query на \`1/sqrt(head_k_dim)\` и возвращает \`(batch, seq_len, num_v_heads, head_v_dim)\`. **Не повторяй** эти шаги.
6. Вычисли \`z = in_proj_z(x)\` из исходных скрытых состояний, измени его форму на ту же, что и у выхода рекуррентной функции. Примени \`self.norm(core_output, z)\` отдельно для каждой value-головы, используя предоставленный \`RMSNormGated\`. Объедини две размерности голов и вызови \`out_proj\` для получения \`(batch, seq_len, hidden_size)\`.

**Сохраняй вычислительный граф** через свёртку, проекции, затухание, рекурсию и gated-нормировку. Последовательная рекуррентная реализация поддерживает backward: не отсоединяй входы через \`detach\` и не оборачивай forward в \`torch.no_grad()\`.

Имена модулей и примеры: \`tests/test_qwen_public.py::TestGatedDeltaNet\`.

Блочная рекуррентность — отдельный необязательный бонус в конце.

In [ ]:
batch, seq_len = 2, 37
heads, k_dim, v_dim = small_config.linear_num_value_heads, small_config.linear_key_head_dim, small_config.linear_value_head_dim
query = torch.randn(batch, seq_len, heads, k_dim)
key = torch.randn(batch, seq_len, heads, k_dim)
value = torch.randn(batch, seq_len, heads, v_dim)
g = -torch.rand(batch, seq_len, heads)
beta = torch.rand(batch, seq_len, heads)

out_recurrent = qwen.gated_delta_rule_recurrent(query, key, value, g, beta)
print("output shape:", out_recurrent.shape)

In [ ]:
layer_idx = small_config.layer_types.index("linear_attention")
gdn = qwen.GatedDeltaNet(small_config, layer_idx)
gdn.eval()
hidden_states = torch.randn(2, 19, small_config.hidden_size)
with torch.no_grad():
    print("GatedDeltaNet output shape:", gdn(hidden_states).shape)

#### 8. Слой декодера: объединяем Attention/GDN и MLP в residual-блок

Pre-LN residual-блок вызывает тот тип attention, который назначен данному слою:

$$
h \leftarrow h + \text{Attn}\big(\text{input\_layernorm}(h)\big)
$$

$$
h \leftarrow h + \text{MLP}\big(\text{post\_attention\_layernorm}(h)\big)
$$

Конструктор предоставлен. Реализуй \`Qwen35DecoderLayer.forward\` в \`qwen.py\`. Точные имена атрибутов смотри в \`tests/test_qwen_public.py::TestDecoderLayer\`.

In [ ]:
layer_idx = small_config.layer_types.index("linear_attention")
layer = qwen.Qwen35DecoderLayer(small_config, layer_idx)
layer.eval()

batch, seq_len = 2, 12
hidden_states = torch.randn(batch, seq_len, small_config.hidden_size)
mask = qwen.build_causal_mask(seq_len, hidden_states.dtype, hidden_states.device)
position_embeddings = qwen.Qwen35RotaryEmbedding(small_config)(
    hidden_states, torch.arange(seq_len).unsqueeze(0).expand(batch, -1)
)
with torch.no_grad():
    out = layer(hidden_states, position_embeddings, mask)
print("output shape:", out.shape, " block_type:", layer.block_type)

#### 9. Полная модель: \`Qwen35Model\` и \`Qwen35ForCausalLM\`

\`Qwen35Model\` размещает стек декодерных слоёв между матрицей эмбеддингов и финальным \`RMSNorm\`. \`Qwen35ForCausalLM\` добавляет голову языковой модели:

$$
\text{logits} = h\, W_{\text{lm\_head}}^\top
$$

$$
\mathcal{L} = \operatorname{CrossEntropy}(\text{logits},\ \text{labels})
$$

\`lm_head\` — настоящий \`nn.Linear\`. При \`tie_word_embeddings=True\` его вес связан с \`embed_tokens.weight\` — численно это то же умножение на общую матрицу. Модели с несвязанными весами (см. другие размеры ниже) поддерживаются так же.

Реализуй \`Qwen35Model.forward\` и \`Qwen35ForCausalLM.forward\` в \`qwen.py\`.

Основа модели:
- Преобразует input IDs в эмбеддинги.
- Строит позиции \`0..seq_len-1\` и аддитивную причинную маску на устройстве входного тензора.
- Пропускает представления через декодерные слои и финальную нормировку.
- Возвращает **тензор скрытых состояний**, а не объект \`transformers\`.

Обёртка LM возвращает \`(logits, loss)\`, причём \`loss=None\` без меток.

**Метки уже выровнены с логитами.** Эта обёртка не сдвигает их автоматически. Для обучения предсказанию следующего токена передавай \`input_ids=tokens[:, :-1]\`, \`labels=tokens[:, 1:]\`; cross-entropy считается на соответствующих позициях. Значение метки \`-100\` означает пропуск соответствующего слагаемого loss (\`ignore_index\` по умолчанию). Такая маска потерь **не** добавляет поддержку padding-масок для внимания.

In [ ]:
model = qwen.Qwen35ForCausalLM(small_config)
model.eval()
tokens = torch.randint(0, small_config.vocab_size, (2, 17))
input_ids, labels = tokens[:, :-1], tokens[:, 1:]
with torch.no_grad():
    logits, loss = model(input_ids, labels=labels)
print("logits shape:", logits.shape)
print("loss:", loss.item())

#### Настоящие веса: загрузка Qwen3.5-0.8B и генерация текста

Этот эксперимент скачивает настоящий чекпоинт `Qwen/Qwen3.5-0.8B`, загружает его в твою `Qwen35ForCausalLM` и сравнивает жадное декодирование с оригинальной реализацией `transformers` при одинаковых весах и токенах.

Это сквозная интеграционная проверка помимо unit-тестов.

При включении эксперимент загружает около **1,6–1,8 ГБ** в `HF_HOME` (по умолчанию `.cache/huggingface`). Устанавливай `RUN_REAL_WEIGHTS_DEMO=True` только после реализации конвертера конфигураций ниже **и бонуса GQA**. В реальном чекпоинте key/value-голов меньше, чем query-голов, поэтому одной обязательной реализации обычного attention недостаточно.

Интеграционные тесты: `tests/test_qwen_real_weights.py`.

In [ ]:
RUN_REAL_WEIGHTS_DEMO = False

if RUN_REAL_WEIGHTS_DEMO:
    from huggingface_hub import snapshot_download
    from safetensors.torch import load_file
    import glob, os

    hf_config = transformers.AutoConfig.from_pretrained("Qwen/Qwen3.5-0.8B")

    real_config = qwen.qwen35_config_from_hf(hf_config.to_dict())
    real_model = qwen.Qwen35ForCausalLM(real_config)

    ckpt_dir = snapshot_download("Qwen/Qwen3.5-0.8B")
    state_dict = {}
    for path in sorted(glob.glob(os.path.join(ckpt_dir, "*.safetensors"))):
        state_dict.update(load_file(path))

    prefix = "model.language_model."
    clean_state_dict = {
        key[len(prefix):]: value for key, value in state_dict.items()
        if key.startswith(prefix) and not key.startswith(prefix + "mtp.")
    }
    real_model.model.load_state_dict(clean_state_dict, strict=True)

    tokenizer = transformers.AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B")
    real_model.eval()
    tokens = tokenizer.encode("The Fermi paradox ")
    for _ in range(60):
        with torch.no_grad():
            logits, _ = real_model(torch.as_tensor([tokens]))
        p_next = torch.softmax(logits[0, -1], dim=-1).numpy()
        tokens.append(int(p_next.argmax()))
    print(tokenizer.decode(tokens))
else:
    print("Skipped (RUN_REAL_WEIGHTS_DEMO=False).")

In [ ]:
from transformers.models.qwen3_5 import modeling_qwen3_5 as ref

if RUN_REAL_WEIGHTS_DEMO:
    real_text_model = ref.Qwen3_5TextModel(hf_config.get_text_config())
    real_text_model.load_state_dict(clean_state_dict, strict=True)
    real_text_model.eval()

    ref_tokens = tokenizer.encode("The Fermi paradox ")
    for _ in range(60):
        with torch.no_grad():
            hidden = real_text_model(torch.as_tensor([ref_tokens])).last_hidden_state
            logits = hidden @ real_text_model.embed_tokens.weight.T
        p_next = torch.softmax(logits[0, -1], dim=-1).numpy()
        ref_tokens.append(int(p_next.argmax()))
    print(tokenizer.decode(ref_tokens))
    assert ref_tokens == tokens, "Greedy token sequences differ from the reference"
else:
    print("Skipped (RUN_REAL_WEIGHTS_DEMO=False).")

#### Масштабирование на другие размеры Qwen3.5

Почти ничего в архитектуре не зависит исключительно от размера 0.8B: плотные версии **0.8B/4B/9B/27B** используют общие `full_attention_interval`, `head_dim`, `rope_theta`, `partial_rotary_factor`. Отличаются размерности и соотношение линейных голов (уже поддержано через `repeat_interleave` в `GatedDeltaNet.forward`).

Все эти чекпоинты также используют GQA: для запуска нужен бонус, но конвертер конфигурации входит в **обязательную** часть. Сохраняй `num_key_value_heads` из чекпоинта, не подменяй его `num_attention_heads`.

Важное отличие: модели 0.8B/4B связывают веса токеновых эмбеддингов и головы LM, а **9B/27B — нет**. Это поддерживает настоящий `nn.Linear` в `Qwen35ForCausalLM`.

Qwen3.5-35B-A3B — другая, MoE-архитектура, в эту работу не входит.

**Реализуй `qwen35_config_from_hf` в `qwen.py`**, преобразующую реальный `config.json` модели в `Qwen35Config`:

- https://huggingface.co/Qwen/Qwen3.5-0.8B/blob/main/config.json
- https://huggingface.co/Qwen/Qwen3.5-4B/blob/main/config.json
- https://huggingface.co/Qwen/Qwen3.5-9B/blob/main/config.json
- https://huggingface.co/Qwen/Qwen3.5-27B/blob/main/config.json

Перенеси **все представленные поля архитектуры**, включая `linear_conv_kernel_dim`, `linear_key_head_dim`, `linear_value_head_dim`, оба количества GDN-голов, epsilon нормировки, activation, attention bias, длину контекста и явно заданные `layer_types`.

Если оставить размерности голов 16 из маленькой конфигурации вместо 128 в реальном чекпоинте, модель будет несовместима. Также отобрази `mtp_num_hidden_layers` в `mtp_num_layers`.

Поддержи полный словарь с `text_config` и словарь только текстовых настроек. Для полного словаря значение `tie_word_embeddings` на верхнем уровне имеет приоритет; иначе бери его из текстовой конфигурации, затем значение dataclass по умолчанию.

Читай RoPE-параметры из `rope_parameters` (или устаревшего `rope_scaling`), а при их отсутствии — из плоских полей/default. Остальные подсказки по именованию есть в docstring `qwen.py`.

In [ ]:
# A literal, offline copy of Qwen3.5-4B's config.json (the parts we need) --
# pasted directly rather than downloaded, so this cell needs no network access.
qwen35_4b_config = {
    "tie_word_embeddings": True,
    "text_config": {
        "vocab_size": 248320, "hidden_size": 2560, "intermediate_size": 9216,
        "num_hidden_layers": 32, "num_attention_heads": 16, "num_key_value_heads": 4,
        "head_dim": 256, "attention_bias": False,
        "linear_conv_kernel_dim": 4, "linear_key_head_dim": 128, "linear_value_head_dim": 128,
        "linear_num_key_heads": 16, "linear_num_value_heads": 32,
        "max_position_embeddings": 262144, "rms_norm_eps": 1e-06, "hidden_act": "silu",
        "full_attention_interval": 4, "mtp_num_hidden_layers": 1,
        "tie_word_embeddings": True,
        "rope_parameters": {"rope_theta": 10000000, "partial_rotary_factor": 0.25},
    },
}

config_4b = qwen.qwen35_config_from_hf(qwen35_4b_config)
print(config_4b)
print("\nnum layer_types:", len(config_4b.layer_types), " full_attention count:",
      config_4b.layer_types.count("full_attention"))

#### Бонус (необязательно): grouped-query attention (GQA)

Реализуй `GatedAttention.forward_gqa` в `qwen.py`. Он расширяет обязательный attention на конфигурации, где `num_key_value_heads` меньше `num_attention_heads` и точно делит его.

Предоставленный `forward` автоматически вызывает метод для этих конфигураций. Маленькая модель использует обычный attention и не требует бонуса.

Проекция возвращает головы query формы `(batch, Hq, seq_len, head_dim)`, а головы key/value — `(batch, Hkv, seq_len, head_dim)`.

Пусть `groups = Hq // Hkv`. Голова query с индексом `h` должна использовать key/value-голову `h // groups`. Например, для четырёх query-голов и двух KV-голов соответствие — `[0, 0, 1, 1]`.

Повтори каждую KV-голову **подряд** по оси голов (либо реализуй эквивалентные групповые батчевые вычисления), затем вычисли масштабированные scores, аддитивную/причинную маску, softmax, sigmoid-затвор и выходную проекцию точно как в обычном attention.

Сохраняй градиенты и векторизацию; **не используй SDPA и не перебирай головы по одной в Python**.

GQA и другие способы ускорения подробно рассматриваются на неделе 7. Этот бонус позволяет запускать пример настоящего чекпоинта выше, поскольку Qwen3.5-0.8B использует GQA. Он независим от бонусов по блочной рекурсии и MTP.

Когда закончишь, включи `RUN_GQA_BONUS=True` ниже.

In [ ]:
RUN_GQA_BONUS = False

if RUN_GQA_BONUS:
    gqa_config = qwen.make_small_config(num_attention_heads=4, num_key_value_heads=2)
    layer_idx = gqa_config.layer_types.index("full_attention")
    gqa = qwen.GatedAttention(gqa_config, layer_idx).eval()
    batch, seq_len = 2, 12
    hidden_states = torch.randn(batch, seq_len, gqa_config.hidden_size)
    mask = qwen.build_causal_mask(seq_len, hidden_states.dtype, hidden_states.device)
    position_embeddings = qwen.Qwen35RotaryEmbedding(gqa_config)(
        hidden_states, torch.arange(seq_len).unsqueeze(0).expand(batch, -1)
    )
    with torch.no_grad():
        out_gqa = gqa(hidden_states, position_embeddings, mask)
        out_naive = qwen.naive_looped_gated_attention(
            gqa, hidden_states, position_embeddings, mask
        )
    print("output shape:", out_gqa.shape)
    print("matches naive per-head loop:", torch.allclose(out_gqa, out_naive, atol=1e-4))
else:
    print("Skipped optional GQA bonus.")


#### Бонус (необязательно): эффективное блочное delta rule

`gated_delta_rule_recurrent` корректен, но перебирает `seq_len` в цикле Python — на длинных последовательностях GPU работает медленно.

Настоящий `transformers` переписывает те же вычисления в виде **блоков (chunks)** по трюку «WY representation»: учитывает взаимодействия внутри блока несколькими матричными умножениями размера блока, затем проходит по блокам, перенося небольшое рекуррентное состояние.

Реализуй `gated_delta_rule_chunked` в `qwen.py`, чтобы результат совпадал с `gated_delta_rule_recurrent` в пределах погрешности float.

Только переход **между блоками** может быть последовательным: нужно `O(seq_len / chunk_size)` обновлений состояния, а **внутриблочные вычисления — батчевые**. Допускается цикл коррекции внутри блока, ограниченный `chunk_size`.

Это самый сложный блок работы: самостоятельно разберись с алгоритмом WY representation. Для ориентира можешь изучить `torch_chunk_gated_delta_rule` в реальном `transformers`. Сравни результаты с `gated_delta_rule_recurrent` на маленькой последовательности, длина которой **не кратна** `chunk_size`.

Эта функция не подключена к `GatedDeltaNet.forward`. Публичные примеры — `tests/test_qwen_bonus.py`.

Простой вызов или копирование покадровой рекурсии не выполняет условия бонуса, даже если результаты корректны и время растёт линейно. Обрабатывай остаток короткого блока, сохраняй состояние через его границы и применяй батчевые вычисления внутри блока. Одного профилирования скорости недостаточно как доказательства.

Включай `RUN_CHUNKED_BONUS=True` только после готовности.

In [ ]:
RUN_CHUNKED_BONUS = False

if RUN_CHUNKED_BONUS:
    batch, seq_len = 2, 37  # deliberately NOT a multiple of chunk_size=64
    heads, k_dim, v_dim = small_config.linear_num_value_heads, small_config.linear_key_head_dim, small_config.linear_value_head_dim
    query = torch.randn(batch, seq_len, heads, k_dim)
    key = torch.randn(batch, seq_len, heads, k_dim)
    value = torch.randn(batch, seq_len, heads, v_dim)
    g = -torch.rand(batch, seq_len, heads)
    beta = torch.rand(batch, seq_len, heads)

    out_chunked = qwen.gated_delta_rule_chunked(query, key, value, g, beta)
    out_recurrent = qwen.gated_delta_rule_recurrent(query, key, value, g, beta)
    print("output shape:", out_chunked.shape)
    print("matches the recurrent reference:", torch.allclose(out_chunked, out_recurrent, atol=1e-3))
else:
    print("Skipped optional chunked bonus.")

#### Бонус (необязательно): голова Multi-Token Prediction (MTP)

Реализуй учебную функцию `Qwen35MTPHead.forward` в `qwen.py`. Для длины последовательности `L >= 3` позиция `t` предсказывает токен `t+2`, используя состояние основы на позиции `t` и **известный** эмбеддинг токена `t+1`.

1. Возьми `hidden_states[:, :-2]` и `embed_tokens(input_ids[:, 1:-1])`. Нормализуй их независимо через `pre_fc_norm_hidden` и `pre_fc_norm_embedding`.
2. Конкатенируй **сначала скрытое состояние, потом эмбеддинг** по последней оси; вызови `fc` и получи признаки `(batch, L-2, hidden_size)`.
3. Построй причинную маску длины `L-2` и RoPE для абсолютных позиций **`1..L-2`**, соответствующих эмбеддингам следующих токенов. Пропусти их через все слои `self.layers` (каждый — полносвязный по вниманию декодер).
4. Примени финальную `norm`, затем умножь на `embed_tokens.weight.T`, получив логиты `(batch, L-2, vocab_size)`. Верни **только logits**. Эталонные цели — `input_ids[:, 2:]`; **не считывай** эмбеддинги этих целевых токенов как входы. Для `L < 3` выбрасывай `ValueError`, так как предсказывать нечего.

Упрощённый бонус использует матрицу эмбеддингов как выходной классификатор даже в случае, когда у основной LM несвязанные веса. Это **не** полная совместимая с чекпоинтом реализация Qwen MTP/speculative decoding.

Конструктор уже предоставлен и поддерживает `mtp_num_layers` декодерных слоёв. Сохраняй градиенты по обоим потокам признаков и всем слоям.

Помимо правильных форм обеспечь точное выравнивание, причинность, заданные вычисления и градиенты. Когда закончишь, включи `RUN_MTP_BONUS=True`. Этот бонус использует `small_config` с равным количеством голов и не зависит от GQA и блочной рекурсии.

In [ ]:
RUN_MTP_BONUS = False

if RUN_MTP_BONUS:
    mtp = qwen.Qwen35MTPHead(small_config)
    model.eval(); mtp.eval()

    batch, seq_len = 2, 16
    input_ids = torch.randint(0, small_config.vocab_size, (batch, seq_len))
    with torch.no_grad():
        hidden_states = model.model(input_ids)
        mtp_logits = mtp(hidden_states, input_ids, model.model.embed_tokens)

    print("mtp logits shape:", mtp_logits.shape, " (expected: (batch, seq_len - 2, vocab_size))")
else:
    print("Skipped optional MTP bonus.")